In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from ultralytics import YOLO
import ultralytics.nn.modules as modules
import ultralytics.nn.modules.block as block
import ultralytics.nn.tasks as tasks

# ============================
# 1. SWIN TRANSFORMER (ĐÃ SỬA)
# ============================
class WindowAttention(nn.Module):
    def __init__(self, dim, window_size, num_heads):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        q = q * self.scale
        attn = (q @ k.transpose(-2, -1))
        attn = self.softmax(attn)
        
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        return x

class SwinTransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, window_size=7, shift_size=0, mlp_ratio=4.0):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size

        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(dim, window_size, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Linear(mlp_hidden_dim, dim)
        )

    def forward(self, x):
        H, W = self.H, self.W
        B, L, C = x.shape
        
        shortcut = x
        x = self.norm1(x)
        x = x.view(B, H, W, C)

        pad_l = pad_t = 0
        pad_r = (self.window_size - W % self.window_size) % self.window_size
        pad_b = (self.window_size - H % self.window_size) % self.window_size
        x = F.pad(x, (0, 0, pad_l, pad_r, pad_t, pad_b))
        _, Hp, Wp, _ = x.shape

        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
        else:
            shifted_x = x

        x_windows = self.window_partition(shifted_x, self.window_size)
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)
        attn_windows = self.attn(x_windows)
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        shifted_x = self.window_reverse(attn_windows, self.window_size, Hp, Wp)

        if self.shift_size > 0:
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            x = shifted_x

        if pad_r > 0 or pad_b > 0:
            x = x[:, :H, :W, :].contiguous()

        x = x.view(B, H * W, C)
        x = shortcut + x
        x = x + self.mlp(self.norm2(x))
        return x

    def window_partition(self, x, window_size):
        B, H, W, C = x.shape
        x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
        windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
        return windows

    def window_reverse(self, windows, window_size, H, W):
        B = int(windows.shape[0] / (H * W / window_size / window_size))
        x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
        x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
        return x

class _SwinTransformerCore(nn.Module):
    def __init__(self, c1, c2, num_heads=4, depth=2, window_size=7):
        super().__init__()
        self.c1 = c1
        self.c2 = c2
        self.window_size = window_size
        
        self.input_proj = nn.Conv2d(c1, c2, kernel_size=1)
        
        self.blocks = nn.ModuleList()
        for i in range(depth):
            shift_size = 0 if (i % 2 == 0) else window_size // 2
            self.blocks.append(
                SwinTransformerBlock(dim=c2, num_heads=num_heads, 
                                   window_size=window_size, shift_size=shift_size)
            )
        
        self.output_proj = nn.Conv2d(c2, c2, kernel_size=1)
        self.norm = nn.LayerNorm(c2)

    def forward(self, x):
        B, C, H, W = x.shape
        x = self.input_proj(x)
        x = x.flatten(2).transpose(1, 2)
        
        for block in self.blocks:
            block.H, block.W = H, W
            x = block(x)
        
        x = self.norm(x)
        x = x.transpose(1, 2).reshape(B, self.c2, H, W)
        x = self.output_proj(x)
        return x

# Wrapper cho Ultralytics (ĐÃ SỬA __init__)
class SwinTransformer(nn.Module):
    """
    Wrapper for Ultralytics compatibility.
    Sửa đổi: Dùng *args để nhận tham số linh hoạt, tránh lỗi missing argument.
    """
    def __init__(self, c1, *args, **kwargs):
        super().__init__()
        self.c1 = c1  # Input channels (Ultralytics tự truyền vào đầu tiên)
        
        # Xử lý args từ YAML [256, num_heads, depth, window_size]
        # args[0] sẽ là c2 (256)
        if len(args) > 0:
            self.c2 = args[0]
        else:
            # Fallback nếu không đọc được args từ YAML (tránh crash)
            print(f"⚠️ Warning: Không tìm thấy c2 trong args. Mặc định c2=c1={c1}")
            self.c2 = c1

        # Các tham số khác (nếu YAML không truyền thì dùng mặc định)
        self.num_heads = args[1] if len(args) > 1 else 4
        self.depth = args[2] if len(args) > 2 else 2
        self.window_size = args[3] if len(args) > 3 else 7

        self.core = None 

    def forward(self, x):
        # Lazy initialization
        if self.core is None:
            self.core = _SwinTransformerCore(
                c1=x.shape[1],  # Lấy kênh thực tế
                c2=self.c2, 
                num_heads=self.num_heads,
                depth=self.depth,
                window_size=self.window_size
            ).to(x.device)
            
        return self.core(x)

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import sys
from ultralytics import YOLO
import ultralytics.nn.modules as modules
import ultralytics.nn.modules.block as block
import ultralytics.nn.tasks as tasks
import torch.nn as nn

# ============================
# 2. ĐĂNG KÝ VÀ CHẠY
# ============================
# Đăng ký class mới
modules.SwinTransformer = SwinTransformer
block.SwinTransformer = SwinTransformer
tasks.SwinTransformer = SwinTransformer

# 3. CẤU HÌNH & LOAD MODEL
# ============================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️ Sử dụng device: {device}")

project_name = 'runs_yolo11s_load11'
exp_name = 'swin_uav_load11'
save_dir = os.path.join(project_name, exp_name)

# Đường dẫn file của bạn
yaml_path = '/home/nguyen/Downloads/UAV_Slicing/final_dataset/caiTien_new.yaml'
data_path = '/home/nguyen/Downloads/UAV_Slicing/dataset.yaml'

# ============================
# LOAD MODEL
# ============================
# Nếu đã có checkpoint -> resume
last_checkpoint = os.path.join(save_dir, "weights", "last.pt")

if os.path.exists(last_checkpoint):
    print(f"🔄 Resume training from checkpoint: {last_checkpoint}")
    model = YOLO(last_checkpoint)     # load đúng last.pt
    resume_flag = True
else:
    print("🚀 Bắt đầu huấn luyện mới...")
    model = YOLO(yaml_path)
    # Load pretrained weights nếu cần
    model.load('yolo11s.pt')  # Uncomment nếu muốn load pretrained
    resume_flag = False

print(f"📄 Model config: {yaml_path}")
print(f"📄 Dataset YAML: {data_path}\n")


if __name__ == '__main__':
    # Load model
    torch.cuda.empty_cache()
    
    # Train
    model.train(
        data=data_path,
        epochs=100,
        optimizer='SGD',
        momentum=0.9,
        lr0=0.001,
        imgsz=640,
        batch=4,
        device=device,
        project=project_name,
        name=exp_name,
        verbose=True,
        amp=False,          # ⚠️ Tắt AMP do GPU T1000 báo lỗi tương thích
        exist_ok=True,
        cache=False,
        resume=resume_flag
    )

🖥️ Sử dụng device: cuda
🔄 Resume training from checkpoint: runs_yolo11s_load11/swin_uav_load11/weights/last.pt
📄 Model config: /home/nguyen/Downloads/UAV_Slicing/final_dataset/caiTien_new.yaml
📄 Dataset YAML: /home/nguyen/Downloads/UAV_Slicing/dataset.yaml

New https://pypi.org/project/ultralytics/8.4.12 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.6 🚀 Python-3.10.12 torch-2.8.0+cu128 CUDA:0 (NVIDIA T1000 8GB, 7967MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/nguyen/Downloads/UAV_Slicing/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=F

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import sys
from ultralytics import YOLO
import ultralytics.nn.modules as modules
import ultralytics.nn.modules.block as block
import ultralytics.nn.tasks as tasks
import torch.nn as nn

# ============================
# 2. ĐĂNG KÝ VÀ CHẠY
# ============================
# Đăng ký class mới
modules.SwinTransformer = SwinTransformer
block.SwinTransformer = SwinTransformer
tasks.SwinTransformer = SwinTransformer

In [4]:
from ultralytics import YOLO

model = YOLO('/home/nguyen/Downloads/UAV_Slicing/runs_yolo11s_load11/swin_uav_load11/weights/best.pt')

metrics = model.val(
    data='/home/nguyen/Downloads/UAV_Slicing/dataset.yaml',
    split='test',
    imgsz=640,
    batch=4,
    device='cuda',
    project='runs_yolo11s_load/swin_uav_load11',           # lưu cùng thư mục train
    name='yolov11_test',          # thư mục kết quả test
    verbose=True
)

print(metrics)


Ultralytics 8.4.6 🚀 Python-3.10.12 torch-2.8.0+cu128 CUDA:0 (NVIDIA T1000 8GB, 7967MiB)
caiTien_new summary (fused): 132 layers, 12,013,060 parameters, 0 gradients, 209.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 354.7±109.4 MB/s, size: 71.7 KB)
val: Scanning /home/nguyen/Downloads/UAV_Slicing/dataset_80_10_10/labels/test.cache... 1000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1000/1000 419.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 250/250 1.4it/s 3:00<0.8s
                   all       1000       1136      0.884      0.846      0.886      0.425
Speed: 0.6ms preprocess, 177.6ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /home/nguyen/Downloads/UAV_Slicing/runs/detect/runs_yolo11s_load/swin_uav_load11/yolov11_test
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix:

In [ ]:
# ============================
# 2. ĐĂNG KÝ VÀ CHẠY
# ============================
# Đăng ký class mới
modules.SwinTransformer = SwinTransformer
block.SwinTransformer = SwinTransformer
tasks.SwinTransformer = SwinTransformer

# Đường dẫn file của bạn
yaml_path = '/home/nguyen/Downloads/UAV_Slicing/final_dataset/caiTien_new.yaml'
data_path = '/home/nguyen/Downloads/UAV_Slicing/dataset.yaml'

device = 'cuda' if torch.cuda.is_available() else 'cpu'

if __name__ == '__main__':
    # Load model
    model = YOLO(yaml_path)
    
    # Traina
    model.train(
        data=data_path,
        epochs=200,
        optimizer='SGD',
        momentum=0.9,
        lr0=0.001,
        imgsz=640,
        batch=16,
        device=device,
        project='runs_yolo11s_swin',
        name='swin_uav_fixed',
        verbose=True
    )